# Predicting Product Sales
## 5. Feature Engineering

Steps:
1. Split the preprocessed dataset into Train and Test subsets (80/20, random_state=42) to prevent data leakage.
2. Extract temporal features: `Month` (string representation), `DayOfWeek`, and `Is_Weekend` (binary).
3. Drop raw `invoice_date`.
4. Perform one-hot encoding on categorical columns.
5. Standard scale numeric age feature.
6. Save training and testing matrices along with preprocessor instances.

In [ ]:
import pandas as pd
import joblib
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
import warnings
warnings.filterwarnings('ignore')

CLEANED_DATA_PATH = r'./data/preprocessed-data/preprocessed_customer_shopping_data.csv'
data = pd.read_csv(CLEANED_DATA_PATH, parse_dates=['invoice_date'])
print('Data loaded. Shape:', data.shape)

### 5.1 Train-Test Split

In [ ]:
X = data.drop(columns=['Sales_Revenue'])
y = data['Sales_Revenue']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print('Train shape:', X_train.shape)
print('Test shape :', X_test.shape)

### 5.2 Extract Temporal Features

In [ ]:
def extract_time_features(df):
    df_copy = df.copy()
    df_copy['Month'] = df_copy['invoice_date'].dt.month.astype(str)
    df_copy['DayOfWeek'] = df_copy['invoice_date'].dt.day_name()
    df_copy['Is_Weekend'] = df_copy['DayOfWeek'].isin(['Saturday', 'Sunday']).astype(int)
    df_copy = df_copy.drop(columns=['invoice_date'])
    return df_copy

X_train = extract_time_features(X_train)
X_test = extract_time_features(X_test)
print('Columns after time feature extraction:', list(X_train.columns))

### 5.3 One-Hot Encoding
Categorical columns are encoded. We fit the encoder on training data only to avoid target leakage.

In [ ]:
categorical_cols = ['gender', 'category', 'payment_method', 'shopping_mall', 'Month', 'DayOfWeek']
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

encoder.fit(X_train[categorical_cols])

X_train_encoded = encoder.transform(X_train[categorical_cols])
X_test_encoded = encoder.transform(X_test[categorical_cols])

feature_names = encoder.get_feature_names_out(categorical_cols)

X_train_enc_df = pd.DataFrame(X_train_encoded, columns=feature_names, index=X_train.index)
X_test_enc_df = pd.DataFrame(X_test_encoded, columns=feature_names, index=X_test.index)

X_train = pd.concat([X_train.drop(columns=categorical_cols), X_train_enc_df], axis=1)
X_test = pd.concat([X_test.drop(columns=categorical_cols), X_test_enc_df], axis=1)

print('Shape after encoding:', X_train.shape)

### 5.4 Feature Scaling
We scale the `age` feature. The scaler is fit on the training data only.

In [ ]:
scaler = StandardScaler()
scaler.fit(X_train[['age']])

X_train['age'] = scaler.transform(X_train[['age']])
X_test['age'] = scaler.transform(X_test[['age']])

X_train['age'].head()

### 5.5 Save Final Datasets

In [ ]:
out_dir = r'./data/ready_for_train'
os.makedirs(out_dir, exist_ok=True)

joblib.dump(X_train, os.path.join(out_dir, 'X_train.pkl'))
joblib.dump(X_test, os.path.join(out_dir, 'X_test.pkl'))
joblib.dump(y_train, os.path.join(out_dir, 'y_train.pkl'))
joblib.dump(y_test, os.path.join(out_dir, 'y_test.pkl'))

joblib.dump(encoder, os.path.join(out_dir, 'onehot_encoder.pkl'))
joblib.dump(scaler, os.path.join(out_dir, 'standard_scaler.pkl'))

print('All matrices and preprocessor instances saved successfully.')